# Video 05 - Exercise Solutions

Try the exercises in [exercises.md](../exercises.md) yourself first. These are **one** good answer each.

In [0]:
import json
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"

def json_format(name, properties):
    """Build a strict response_format where every property is required."""
    return {
        "type": "json_schema",
        "json_schema": {
            "name": name,
            "strict": True,
            "schema": {"type": "object", "properties": properties,
                       "required": list(properties), "additionalProperties": False},
        },
    }

def ask_json(system, user, response_format):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format=response_format,
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)

In [0]:
review_format = json_format("review", {
    "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
    "confidence": {"type": "number"},
    "topic": {"type": "string", "enum": ["battery", "delivery", "price", "support", "other"]},
})

for review in [
    "Arrived late and the box was crushed",
    "Battery lasts all week, amazing",
    "Way too expensive for what it is",
    "Nobody answered my emails for a week",
]:
    print(f"{review:40} -> {ask_json('Classify the review.', review, review_format)}")

In [0]:
intent_format = json_format("intent", {
    "intent": {"type": "string", "enum": ["book_table", "cancel_booking", "ask_question"]},
    "people": {"type": ["integer", "null"]},
    "time": {"type": ["string", "null"]},
})
system = "Work out what the customer wants. Use null for details they did not give. Never guess."

for request in ["Book me a table for 2 at 7pm", "Cancel my booking please", "Do you have vegan options?"]:
    print(f"{request:32} -> {ask_json(system, request, intent_format)}")

# The model chose an ACTION (intent) and its ARGUMENTS (people, time).
# That is exactly what tool calling is - Video 06.

In [0]:
int_format = json_format("sentiment_int", {
    "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
    "confidence": {"type": "integer"},   # 0-100 this time
})

def check_sentiment(d):
    """Return a list of problems with a sentiment result (empty list = looks fine)."""
    problems = []
    if d.get("sentiment") not in ("positive", "negative", "neutral"):
        problems.append(f"unexpected label {d.get('sentiment')!r}")
    conf = d.get("confidence")
    if not isinstance(conf, int):
        problems.append(f"confidence is {type(conf).__name__}, expected int")
    elif not 0 <= conf <= 100:
        problems.append(f"confidence {conf} is outside 0-100")
    return problems

for review in ["Love it", "Hate it", "It's a phone"]:
    result = ask_json("Classify the review. confidence is a percentage from 0 to 100.", review, int_format)
    print(f"{review:14} -> {result} | problems: {check_sentiment(result) or 'none'}")

# In our test every answer passed the checker - but confidence was 100 for all three,
# even the neutral "It's a phone". A schema controls the SHAPE, not whether the number is honest.
# Treat model-reported confidence as a hint, never as a measurement.

In [0]:
PLACEHOLDERS = {"", "null", "none", "<unknown>", "unknown", "n/a", "na", "not provided", "not specified"}

def clean(data):
    """Treat placeholder strings (any capitalisation, any surrounding spaces) as missing."""
    return {
        k: None if isinstance(v, str) and v.strip().lower() in PLACEHOLDERS else v
        for k, v in data.items()
    }

print(clean({"time": "N/A", "date": "  ", "name": "Priya", "note": "Unknown", "people": 4}))
# Treating a suspicious value as MISSING means you ask the user, instead of
# booking a table at the time "N/A". Asking costs one message; guessing can cost a customer.